In [ ]:
import numpy as np
import pandas as pd
from pykalman import KalmanFilter
import vectorbt as vbt
import yfinance as yf

### Coke vs Pepsi

In [7]:

pairs = ["KO", "PEP"]
df3 = yf.download(pairs, start="2015-05-05", end="2025-05-05", progress=False)

try:
  stock_a = df3["Close"]["KO"]
  stock_b = df3["Close"]["PEP"]
except (KeyError, TypeError):
  stock_a = df3.xs("Close", level=0, axis=1)["KO"]
  stock_b = df3.xs("Close", level=0, axis=1)["PEP"]

price_data = pd.DataFrame({"KO": stock_a, "PEP": stock_b}).dropna()
s_a = price_data["KO"]
s_b = price_data["PEP"]


delta = 1e-3
trans_cov = delta / (1 - delta) * np.eye(2)
obs_mat = np.vstack([s_b.values, np.ones(len(s_b))]).T[:, np.newaxis, :]

kf = KalmanFilter(
    n_dim_obs=1,
    n_dim_state=2,
    initial_state_mean=[0, 0],
    initial_state_covariance=np.ones((2, 2)),
    transition_matrices=np.eye(2),
    observation_matrices=obs_mat,
    observation_covariance=1.0,
    transition_covariance=trans_cov,
)

state_means, _ = kf.filter(s_a.values)
beta = state_means[:, 0]
alpha = state_means[:, 1]


spread = s_a - (beta * s_b + alpha)
spread_series = pd.Series(spread, index=s_a.index)

window = 60  
rolling_mean = spread_series.rolling(window=window).mean()
rolling_std = spread_series.rolling(window=window).std()
zscore = (spread_series - rolling_mean) / rolling_std

entry_threshold = 2.5
stop_loss_threshold = 4.0


long_spread = zscore < -entry_threshold
short_spread = zscore > entry_threshold
exit_cond = (zscore.abs() <= 0.2) | (zscore.abs() >= stop_loss_threshold)

entries = pd.DataFrame({"KO": long_spread, "PEP": short_spread}, index=s_a.index)

short_entries = pd.DataFrame(
    {"KO": short_spread, "PEP": long_spread}, index=s_a.index
)

exits = pd.DataFrame({"KO": exit_cond, "PEP": exit_cond}, index=s_a.index)

short_exits = pd.DataFrame({"KO": exit_cond, "PEP": exit_cond}, index=s_a.index)


backtest = vbt.Portfolio.from_signals(
    close=price_data,
    entries=entries,
    exits=exits,
    short_entries=short_entries,
    short_exits=short_exits,
    init_cash=10000.0,
    fees=0.0002, 
    cash_sharing=True,
    group_by=True,
    freq="1D",
)

print(backtest.stats())
backtest.plot().show()

Start                            2015-05-05 00:00:00
End                              2025-05-02 00:00:00
Period                            2515 days 00:00:00
Start Value                                  10000.0
End Value                                16633.24516
Total Return [%]                           66.332452
Benchmark Return [%]                      114.667154
Max Gross Exposure [%]                    352.488587
Total Fees Paid                           629.514421
Max Drawdown [%]                           21.423799
Max Drawdown Duration             1061 days 00:00:00
Total Trades                                      77
Total Closed Trades                               77
Total Open Trades                                  0
Open Trade PnL                                   0.0
Win Rate [%]                               49.350649
Best Trade [%]                             19.982684
Worst Trade [%]                            -9.102116
Avg Winning Trade [%]                       3.

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data

